# Notebook 2: Gymnasium Classic Control Environments

This notebook introduces Gymnasium Classic Control environments.

We will:

- Install and import Gymnasium
- List Classic Control environments
- Load each environment
- Understand state space and action space
- Run a random agent
- Record and display video
- Compare environments in a summary table

Classic Control environments are useful for learning basic Reinforcement Learning concepts.

## 1. Install Required Libraries

The next code cell installs the **Gymnasium** library together with its `classic-control` extra dependencies, plus the video tools needed later (`moviepy`, `imageio`, `imageio-ffmpeg`).

**What the code does:**
- Uses a Jupyter shell command (`!pip install ...`) to install packages from PyPI.
- The line is commented out by default — uncomment it if you are on Google Colab, or if Gymnasium is not installed locally yet.
- Run this cell **only once** per environment. After installation you can re-comment it.

In [9]:
# ============================================================
# CELL: Install Gymnasium Classic Control Libraries
# ------------------------------------------------------------
# This cell installs the Gymnasium library with the
# 'classic-control' extra, along with video recording tools.
#   - gymnasium[classic-control] : provides classic RL envs
#   - moviepy                    : helps render/save videos
#   - imageio / imageio-ffmpeg   : video encoding backends
# The '!' prefix runs the command as a shell command.
# Lines are commented out by default; uncomment to install.
# ============================================================

# Install Gymnasium classic control environments
# Uncomment and run if needed

!pip install "gymnasium[classic-control]" moviepy imageio imageio-ffmpeg

  Using cached moviepy-2.2.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl.metadata (1.5 kB)
  Using cached proglog-0.1.12-py3-none-any.whl.metadata (794 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached moviepy-2.2.1-py3-none-any.whl (129 kB)
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.0 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.0 MB 2.1 MB/s eta 0:00:03
   ------- -------------------------------- 1.3/7.0 MB 2.4 MB/s eta 0:00:03
   ------------ --------------------------- 2.1/7.0 MB 2.6 MB/s eta 0:00:02
   ---------------- ----------------------- 2.9/7.0 MB 2.8 MB/s eta 0:00:02
   --------------------- ------------------ 3.7/7.0 MB 3.0 MB/s eta 0:00:02
   ------------------------- -------------- 4.5/7.0 MB 3.2 MB/s eta 0:00:01
   ------------------------------- -------- 5.5/7.0 MB 3.3 MB/s eta 0:00:01

  You can safely remove it manually.


## 2. Import Libraries

The next code cell imports every library used in this notebook. Each import has a specific role:

- `os` — file system utilities used to create the `videos` folder and to list the saved `.mp4` files.
- `gymnasium as gym` — the main RL environment library (successor of OpenAI `gym`). All environments are created via `gym.make(env_id)`.
- `pandas as pd` — used later to build a clean summary table (`DataFrame`) comparing all environments.
- `Video, display` from `IPython.display` — used to embed and display the recorded mp4 videos directly inside the Jupyter notebook.

In [10]:
# ============================================================
# CELL: Import All Required Libraries
# ------------------------------------------------------------
# This cell brings in every library used throughout the
# notebook. Each import has a specific role:
#   - os                    : work with the file system
#                             (create folders, list files,
#                             join paths for saved videos)
#   - gymnasium (as gym)    : main RL library that provides
#                             the Classic Control environments
#   - pandas (as pd)        : used later to build a tidy
#                             summary table (DataFrame) that
#                             compares all environments
#   - IPython.display       : Video and display let us embed
#                             recorded mp4 videos directly
#                             inside the Jupyter notebook
# ============================================================

import os                                   # File system utilities (create video folder, list files)
import gymnasium as gym                     # Gymnasium = the RL environment library (successor of OpenAI gym)
import pandas as pd                         # Pandas = used to create the summary table of environments
from IPython.display import Video, display  # To embed and display recorded videos inside the notebook

## 3. Define Classic Control Environments

The next code cell creates a Python list containing the names (env IDs) of the five **Classic Control** environments we will work with.

**What the code does:**
- Stores the env-IDs as strings inside the list `classic_control_envs`.
- The five environments are:
  - `CartPole-v1` — balance a pole on a moving cart.
  - `MountainCar-v0` — drive an under-powered car up a hill (discrete actions).
  - `MountainCarContinuous-v0` — same task, continuous force.
  - `Pendulum-v1` — swing up and balance a pendulum (continuous torque).
  - `Acrobot-v1` — swing a two-link robot above a target line.
- The variable name on the last line tells Jupyter to print the list as the cell output.

In [11]:
# ============================================================
# CELL: Define the List of Classic Control Environments
# ------------------------------------------------------------
# We store the names (env_id strings) of the five classic
# control environments inside a Python list. Later cells
# loop over this list so we can inspect / record / compare
# them all using a single piece of code.
#
# The five environments are:
#   1. CartPole-v1               -> balance a pole on a cart
#   2. MountainCar-v0            -> drive a car up a hill (discrete actions)
#   3. MountainCarContinuous-v0  -> same task, continuous force
#   4. Pendulum-v1               -> swing up a pendulum
#   5. Acrobot-v1                -> swing a 2-link robot above a line
#
# Writing the variable name alone on the last line tells
# Jupyter to print the list as the cell's output.
# ============================================================

classic_control_envs = [
    "CartPole-v1",               # Balancing problem, discrete actions
    "MountainCar-v0",            # Hill climbing, discrete actions
    "MountainCarContinuous-v0",  # Hill climbing, continuous actions
    "Pendulum-v1",               # Swing up, continuous torque
    "Acrobot-v1"                 # Two-link robot, discrete actions
]

classic_control_envs  # Display the list as the cell output

['CartPole-v1',
 'MountainCar-v0',
 'MountainCarContinuous-v0',
 'Pendulum-v1',
 'Acrobot-v1']

## 4. Helper Function to Explain State Space and Action Space

The next code cell defines a helper function `explain_env(env_id)` that prints the most important information about any Gymnasium environment.

**What the code does:**
- Creates the environment via `gym.make(env_id)`.
- Prints a clearly formatted header.
- Prints the `observation_space` (the *state space* — what the agent sees).
- Prints the `action_space` (what the agent can do — `Discrete` or `Box`).
- Safely reads `reward_range` from `env.unwrapped` using `getattr` (returns `"Not specified"` if missing).
- Prints `max_episode_steps` from `env.spec` (how long episodes can last).
- Closes the environment to release underlying resources.

In [12]:
# ============================================================
# CELL: Helper Function -> explain_env(env_id)
# ------------------------------------------------------------
# This function takes the name (env_id) of any Gymnasium
# environment and prints out the most important details
# we need to understand it as RL practitioners:
#
#   1. Observation / state space  -> what the agent SEES
#   2. Action space               -> what the agent CAN DO
#   3. Reward range               -> bounds of the reward
#   4. Max episode steps          -> how long an episode can run
#
# Steps inside the function:
#   - Create (instantiate) the environment with gym.make()
#   - Print a header line for readability
#   - Print observation_space and action_space attributes
#   - Safely read reward_range from env.unwrapped using
#     getattr() (returns 'Not specified' if missing)
#   - Print max_episode_steps from the environment's spec
#   - Close the environment to release resources
# ============================================================

def explain_env(env_id):
    # Create the Gymnasium environment from its string ID
    env = gym.make(env_id)

    # ---- Print a nicely formatted header ----
    print("=" * 70)                # Top separator line (70 equals signs)
    print("Environment:", env_id)  # Show which environment we are inspecting
    print("=" * 70)                # Bottom separator line

    # ---- Observation (state) space ----
    # observation_space describes WHAT the agent observes
    # at each step (e.g. Box of 4 floats for CartPole).
    print("Observation space / State space:")
    print(env.observation_space)

    # ---- Action space ----
    # action_space describes WHAT actions are available
    # (Discrete(n) means n choices, Box means continuous).
    print("\nAction space:")
    print(env.action_space)

    # ---- Reward range ----
    # Some envs declare a reward_range; some don't.
    # getattr() lets us read it safely with a default.
    print("\nReward range:")
    reward_range = getattr(env.unwrapped, "reward_range", "Not specified")
    print(reward_range)

    # ---- Environment specification ----
    # env.spec.max_episode_steps = step limit per episode.
    print("\nEnvironment specification:")
    print("Max episode steps:", env.spec.max_episode_steps)

    # Close the environment to free underlying resources
    env.close()

## 5. Display Information for All Classic Control Environments

The next code cell loops over every env-ID in `classic_control_envs` and calls the `explain_env` helper defined above.

**What the code does:**
- Iterates through the list of env-IDs.
- Calls `explain_env(env_id)` for each, printing its state space, action space, reward range and max episode steps.

**Important key:**
- `Box(...)` in the output means a **continuous** space.
- `Discrete(n)` means a **finite** set of `n` choices.

In [13]:
# ============================================================
# CELL: Display Information for All Classic Control Envs
# ------------------------------------------------------------
# We loop over every environment name in classic_control_envs
# and call the helper function explain_env() that we defined
# earlier. This prints the state space, action space, reward
# range and max episode steps for ALL environments at once.
#
#   - 'Box'      in the output means a continuous space
#   - 'Discrete' in the output means a finite set of choices
# ============================================================

# Iterate through each environment ID in the list
for env_id in classic_control_envs:
    explain_env(env_id)   # Call the helper to print info about this env

Environment: CartPole-v1
Observation space / State space:
Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

Action space:
Discrete(2)

Reward range:
Not specified

Environment specification:
Max episode steps: 500
Environment: MountainCar-v0
Observation space / State space:
Box([-1.2  -0.07], [0.6  0.07], (2,), float32)

Action space:
Discrete(3)

Reward range:
Not specified

Environment specification:
Max episode steps: 200
Environment: MountainCarContinuous-v0
Observation space / State space:
Box([-1.2  -0.07], [0.6  0.07], (2,), float32)

Action space:
Box(-1.0, 1.0, (1,), float32)

Reward range:
Not specified

Environment specification:
Max episode steps: 999
Environment: Pendulum-v1
Observation space / State space:
Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)

Action space:
Box(-2.0, 2.0, (1,), float32)

Reward range:
Not specified

Environment specification:
Max episode steps: 200
Environment: Acrobot-v1
Observa

## 6. Create Summary Table

The next code cell builds a clean **pandas DataFrame** that compares all environments side-by-side.

**What the code does:**
- Starts with an empty list `summary_rows`.
- Loops over each env-ID, creates the environment, and appends a dictionary containing: name, observation space, action space, max steps, and reward range.
- Closes each environment to free resources.
- Converts the list of dictionaries to a `pandas.DataFrame` and displays it as a tidy table — much easier to compare than the multi-line prints above.

In [14]:
# ============================================================
# CELL: Create a Pandas Summary Table of All Environments
# ------------------------------------------------------------
# Reading printed text for each environment can be hard to
# compare. So here we build a clean comparison TABLE using
# a list-of-dicts -> pandas.DataFrame pattern.
#
# Steps:
#   1. Start with an empty list (summary_rows)
#   2. Loop over each environment name
#   3. Create the environment with gym.make()
#   4. Append a dictionary with key fields:
#        - Environment name
#        - State / observation space (stringified)
#        - Action space (stringified)
#        - Max episode steps
#        - Reward range (safely read with getattr)
#   5. Close the environment to free resources
#   6. Convert summary_rows -> pandas DataFrame
#   7. Display the DataFrame (last line of cell)
# ============================================================

summary_rows = []   # Will hold one dictionary per environment

# Loop through each environment name we want to compare
for env_id in classic_control_envs:
    # Create the environment so we can read its attributes
    env = gym.make(env_id)

    # Append a dictionary for this environment's row
    summary_rows.append({
        "Environment": env_id,                                                            # Name of the environment
        "State / Observation Space": str(env.observation_space),                          # Observation space (as string)
        "Action Space": str(env.action_space),                                            # Action space (as string)
        "Max Episode Steps": env.spec.max_episode_steps,                                  # Episode step limit
        "Reward Range": str(getattr(env.unwrapped, "reward_range", "Not specified"))      # Reward range (safe read)
    })

    # Always close the environment to release resources
    env.close()

# Convert the list of dictionaries to a Pandas DataFrame (each dict becomes one row)
summary_df = pd.DataFrame(summary_rows)

# Display the DataFrame as the cell output (Jupyter renders it as a nice table)
summary_df

,Environment,State / Observation Space,Action Space,Max Episode Steps,Reward Range
0,CartPole-v1,Box([-4.8 -inf -0.41887903 ...,Discrete(2),500,Not specified
1,MountainCar-v0,"Box([-1.2 -0.07], [0.6 0.07], (2,), float32)",Discrete(3),200,Not specified
2,MountainCarContinuous-v0,"Box([-1.2 -0.07], [0.6 0.07], (2,), float32)","Box(-1.0, 1.0, (1,), float32)",999,Not specified
3,Pendulum-v1,"Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)","Box(-2.0, 2.0, (1,), float32)",200,Not specified
4,Acrobot-v1,Box([ -1. -1. -1. -1. ...,Discrete(3),500,Not specified


## 7. Helper Function to Record Random Agent Video

The next code cell defines `record_random_agent_video(env_id, video_folder, max_steps)` which runs a **random agent** in an environment and saves the rollout as an mp4 video.

**What the code does:**
1. Ensures the output folder exists (creates it if not).
2. Creates the environment with `render_mode="rgb_array"` so frames can be captured.
3. Wraps the env with `gym.wrappers.RecordVideo` so every episode is saved to disk.
4. Calls `env.reset(seed=42)` for reproducibility.
5. Loops up to `max_steps`:
   - Samples a random action via `env.action_space.sample()`.
   - Calls `env.step(action)` and accumulates reward.
   - Stops early if `terminated` or `truncated` is `True`.
6. Closes the env (flushes the mp4 to disk) and prints rollout stats.
7. Finds the newest mp4 file for this env and returns its path.

**A random agent** does **not** learn — it simply samples uniformly from the action space. It is a useful baseline before training real RL agents.

In [15]:
# ============================================================
# CELL: Helper Function -> record_random_agent_video(...)
# ------------------------------------------------------------
# This function runs a RANDOM agent inside a Gymnasium env
# and records the entire run as an mp4 video file so we can
# later watch how the environment behaves.
#
# A "random agent" doesn't learn — it just calls
# env.action_space.sample() at every step.
#
# Arguments:
#   - env_id       : name of the Gymnasium environment
#   - video_folder : folder where mp4 files will be saved
#   - max_steps    : maximum number of steps to run
#
# Steps inside the function:
#   1. Create the output folder (if it does not exist).
#   2. Create env with render_mode='rgb_array'
#      (needed so frames can be captured for the video).
#   3. Wrap env with gym.wrappers.RecordVideo to capture
#      every episode into the video folder.
#   4. Reset the environment to get the initial observation.
#   5. Loop up to max_steps:
#         - sample a random action
#         - call env.step(action)
#         - accumulate reward
#         - stop early if terminated or truncated
#   6. Close the environment (flushes the video file to disk).
#   7. Print episode statistics.
#   8. Find the latest mp4 file for this env and return path.
# ============================================================

def record_random_agent_video(env_id, video_folder="videos", max_steps=300):
    # Make sure the output folder exists; create it if needed
    os.makedirs(video_folder, exist_ok=True)

    # render_mode='rgb_array' returns frames as numpy arrays,
    # which is required so the RecordVideo wrapper can save them
    env = gym.make(env_id, render_mode="rgb_array")

    # Wrap the environment with RecordVideo so every episode
    # is automatically saved as an mp4 file in 'video_folder'
    env = gym.wrappers.RecordVideo(
        env,
        video_folder=video_folder,
        name_prefix=env_id,                            # Use env_id as the file name prefix
        episode_trigger=lambda episode_id: True        # Record every episode (always True)
    )

    # Reset the environment to get the very first observation
    # We use seed=42 for reproducibility (same start each run)
    observation, info = env.reset(seed=42)

    # Keep track of the cumulative reward across the episode
    total_reward = 0

    # Run the random agent for at most 'max_steps' steps
    for step in range(max_steps):
        # Random action sampled from the env's action space
        action = env.action_space.sample()

        # Take a step in the environment using that action
        # step() returns: next obs, reward, terminated, truncated, info
        observation, reward, terminated, truncated, info = env.step(action)

        # Accumulate the reward for reporting
        total_reward += reward

        # Stop early if the episode finishes (success or fail)
        if terminated or truncated:
            break

    # Close the env -> this flushes/finishes writing the video
    env.close()

    # Print a small summary of the random rollout
    print(f"Environment: {env_id}")
    print(f"Steps completed: {step + 1}")
    print(f"Total reward: {total_reward}")

    # ---- Find the most recently created mp4 file for this env ----
    video_files = [
        os.path.join(video_folder, f)
        for f in os.listdir(video_folder)
        if f.endswith(".mp4") and f.startswith(env_id)
    ]

    # Pick the newest video file (max by creation time)
    latest_video = max(video_files, key=os.path.getctime)

    # Return the path so the caller can display the video
    return latest_video

## 8. Record and Display Video: CartPole-v1

The next code cell runs a random agent in **CartPole-v1** and displays the recorded video.

**CartPole-v1 facts:**
- State space: continuous, 4 floats (cart position, cart velocity, pole angle, pole angular velocity).
- Action space: discrete with 2 actions (`0` = push cart left, `1` = push cart right).
- Goal: keep the pole balanced as long as possible.
- Reward: `+1` for every step the pole remains balanced.

**What the code does:**
- Calls `record_random_agent_video("CartPole-v1", max_steps=300)` and stores the returned path.
- Embeds the resulting mp4 in the notebook with `display(Video(video_path, embed=True))`.

In [16]:
# ============================================================
# CELL: Record & Display a Random Agent in CartPole-v1
# ------------------------------------------------------------
# CartPole-v1:
#   - Continuous state space (4 floats: cart pos, cart vel,
#     pole angle, pole angular velocity)
#   - Discrete action space {0=push left, 1=push right}
#   - Goal: balance the pole for as long as possible
#   - +1 reward for every step the pole stays balanced
#
# We call our helper to run a random agent and record a video,
# then use IPython's Video() helper to embed it in the notebook.
# ============================================================

# Run a random agent in CartPole-v1 and record the resulting video
video_path = record_random_agent_video("CartPole-v1", max_steps=300)

# Embed and display the recorded mp4 right inside the notebook
display(Video(video_path, embed=True))

c:\Users\KGoel\AppData\Local\Programs\Python\Python312\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at d:\OneDrive - Nextgen Healthcare, LLC\My\PESIT\Sem2\RL\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Environment: CartPole-v1
Steps completed: 26
Total reward: 26.0


In [17]:
# pip install "gymnasium[other]

## 9. Record and Display Video: MountainCar-v0

The next code cell runs a random agent in **MountainCar-v0** and displays the recorded video.

**MountainCar-v0 facts:**
- State space: continuous, 2 floats (car position, car velocity).
- Action space: discrete with 3 actions (`0` = accelerate left, `1` = do nothing, `2` = accelerate right).
- Goal: drive the under-powered car up the right hill.
- Reward: `-1` every step (so reaching the goal faster = less negative total).

**What the code does:**
- Calls the helper to run and record a random rollout (max 300 steps).
- Embeds the mp4 in the notebook. A random agent almost always fails — illustrating why exploration matters in RL.

In [18]:
# ============================================================
# CELL: Record & Display a Random Agent in MountainCar-v0
# ------------------------------------------------------------
# MountainCar-v0:
#   - Continuous state space (position, velocity)
#   - Discrete action space {0=left, 1=no-op, 2=right}
#   - Goal: drive the car up the right hill
#   - Reward = -1 every step (so faster = better)
#
# A purely random agent almost always fails (it lacks the
# coordinated back-and-forth required), but watching it is
# very instructive to see why exploration matters in RL.
# ============================================================

# Run a random agent in MountainCar-v0 and record the video
video_path = record_random_agent_video("MountainCar-v0", max_steps=300)

# Embed and display the recorded mp4 inside the notebook
display(Video(video_path, embed=True))

Environment: MountainCar-v0
Steps completed: 200
Total reward: -200.0


## 10. Record and Display Video: MountainCarContinuous-v0

The next code cell runs a random agent in **MountainCarContinuous-v0** and displays the recorded video.

**MountainCarContinuous-v0 facts:**
- State space: continuous, 2 floats (position, velocity).
- Action space: **continuous** `Box(-1.0, 1.0, (1,), float32)` — a real-valued force in `[-1, +1]`.
- Same task as MountainCar-v0 but with a continuous force.
- Reward is shaped (penalizes large actions, rewards reaching the goal).

**What the code does:**
- Records a random rollout (sampled from a continuous Box) and embeds the video in the notebook.

In [19]:
# ============================================================
# CELL: Record & Display a Random Agent in MountainCarContinuous-v0
# ------------------------------------------------------------
# MountainCarContinuous-v0:
#   - Continuous state space (position, velocity)
#   - CONTINUOUS action space: Box(-1.0, 1.0, (1,), float32)
#     -> the agent picks a real-valued force in [-1, +1]
#   - Goal: drive the car up the right hill
#   - Reward is shaped (penalizes large actions, rewards
#     reaching the goal)
#
# This environment is the same task as MountainCar-v0 but with
# a continuous action space, which is closer to most real-world
# control problems.
# ============================================================

# Run a random agent in MountainCarContinuous-v0 and record the video
video_path = record_random_agent_video("MountainCarContinuous-v0", max_steps=300)

# Embed and display the recorded mp4 inside the notebook
display(Video(video_path, embed=True))


Environment: MountainCarContinuous-v0
Steps completed: 300
Total reward: -9.5863317990844


## 11. Record and Display Video: Pendulum-v1

The next code cell runs a random agent in **Pendulum-v1** and displays the recorded video.

**Pendulum-v1 facts:**
- State space: continuous, 3 floats — `cos(theta)`, `sin(theta)`, angular velocity.
- Action space: continuous `Box(-2.0, 2.0, (1,), float32)` — apply a torque in `[-2, +2]`.
- Goal: swing the pendulum to upright (theta = 0) and keep it there.
- Reward is negative (penalty for distance from upright and for large actions).

**What the code does:**
- Records a random rollout (random torques) and embeds the video. A random agent will swing the pendulum chaotically and rarely reach upright.

In [20]:
# ============================================================
# CELL: Record & Display a Random Agent in Pendulum-v1
# ------------------------------------------------------------
# Pendulum-v1:
#   - Continuous state space: 3 floats
#     (cos(theta), sin(theta), angular velocity)
#   - CONTINUOUS action space: Box(-2.0, 2.0, (1,), float32)
#     -> apply a torque between -2 and +2
#   - Goal: swing the pendulum to upright (theta = 0) and
#     keep it there.
#   - Reward is negative (penalty); the closer to upright
#     and the less torque, the better.
#
# A random agent will swing the pendulum chaotically without
# reaching the upright position.
# ============================================================

# Run a random agent in Pendulum-v1 and record the video
video_path = record_random_agent_video("Pendulum-v1", max_steps=300)

# Embed and display the recorded mp4 inside the notebook
display(Video(video_path, embed=True))

Environment: Pendulum-v1
Steps completed: 200
Total reward: -1222.2535962399377


## 12. Record and Display Video: Acrobot-v1

The next code cell runs a random agent in **Acrobot-v1** and displays the recorded video.

**Acrobot-v1 facts:**
- State space: continuous, 6 floats (angles and angular velocities of the two links).
- Action space: discrete with 3 actions:
  - `0` = apply negative torque
  - `1` = apply zero torque
  - `2` = apply positive torque
- Goal: swing the lower link above a target height line.
- Reward: `-1` every step until success.

**What the code does:**
- Records a random rollout and embeds the video, showing the under-actuated swing-up behaviour.

In [21]:
# ============================================================
# CELL: Record & Display a Random Agent in Acrobot-v1
# ------------------------------------------------------------
# Acrobot-v1:
#   - Continuous state space: 6 floats describing the angles
#     and angular velocities of the two links.
#   - Discrete action space (3 choices):
#       0 = apply negative torque on the joint
#       1 = apply zero torque
#       2 = apply positive torque
#   - Goal: swing the lower link above a given height line.
#   - Reward = -1 every step until success.
#
# Watching the random agent helps us understand the
# under-actuated nature of the task.
# ============================================================

# Run a random agent in Acrobot-v1 and record the video
video_path = record_random_agent_video("Acrobot-v1", max_steps=300)

# Embed and display the recorded mp4 inside the notebook
display(Video(video_path, embed=True))

Environment: Acrobot-v1
Steps completed: 300
Total reward: -300.0


## 13. Understand Discrete vs Continuous Action Space

The next code cell loops over each environment and prints whether its **action space** is discrete or continuous, with the relevant details.

**What the code does:**
- For each env, calls `isinstance(env.action_space, gym.spaces.Discrete)` to detect discrete action spaces, then prints `.n` (number of actions).
- Otherwise, checks for `gym.spaces.Box` and prints `.low` and `.high` (the continuous bounds).

**Why this matters:**
- Discrete actions (e.g. CartPole) -> algorithms like DQN, REINFORCE.
- Continuous actions (e.g. Pendulum) -> algorithms like DDPG, PPO with Gaussian policy, SAC.

In [22]:
# ============================================================
# CELL: Inspect Action Space (Discrete vs Continuous)
# ------------------------------------------------------------
# For each environment we check the TYPE of its action_space:
#   - gym.spaces.Discrete  -> finite set of choices,
#                             use .n to get the count.
#   - gym.spaces.Box       -> continuous (real numbers),
#                             use .low and .high for ranges.
#
# This is important because RL algorithms differ depending
# on whether actions are discrete (e.g. DQN, REINFORCE) or
# continuous (e.g. DDPG, PPO with Gaussian policy, SAC).
# ============================================================

# Loop through each environment to check its action space type
for env_id in classic_control_envs:
    env = gym.make(env_id)                          # Create environment

    print(f"Environment: {env_id}")                 # Print which env we're inspecting

    # Check whether the action space is Discrete
    if isinstance(env.action_space, gym.spaces.Discrete):
        print("Action space type: Discrete")
        print("Number of actions:", env.action_space.n)         # n = number of discrete actions
    # Otherwise check whether it is a continuous Box
    elif isinstance(env.action_space, gym.spaces.Box):
        print("Action space type: Continuous")
        print("Low:", env.action_space.low)                     # Lower bound(s) of action
        print("High:", env.action_space.high)                   # Upper bound(s) of action

    print("-" * 60)                                 # Separator line for readability
    env.close()                                     # Release the environment

Environment: CartPole-v1
Action space type: Discrete
Number of actions: 2
------------------------------------------------------------
Environment: MountainCar-v0
Action space type: Discrete
Number of actions: 3
------------------------------------------------------------
Environment: MountainCarContinuous-v0
Action space type: Continuous
Low: [-1.]
High: [1.]
------------------------------------------------------------
Environment: Pendulum-v1
Action space type: Continuous
Low: [-2.]
High: [2.]
------------------------------------------------------------
Environment: Acrobot-v1
Action space type: Discrete
Number of actions: 3
------------------------------------------------------------


## 14. Understand State / Observation Space

The next code cell loops over each environment and prints its **observation (state) space** in detail.

**What the code does:**
- Prints `env.observation_space` directly.
- If the space is a `Box` (continuous), it also prints `.low`, `.high` and `.shape` — i.e. the bounds of each component and how many components per observation.

**Example (CartPole-v1) — the 4 components are:**
1. cart position
2. cart velocity
3. pole angle
4. pole angular velocity

In [23]:
# ============================================================
# CELL: Inspect State / Observation Space
# ------------------------------------------------------------
# Each environment also has an observation_space describing
# what the agent perceives at each step. For Classic Control
# envs this is typically a Box (continuous) with real-valued
# components.
#
# For each env we print:
#   - The observation_space (as a Box description)
#   - The low and high bounds (limits of each component)
#   - The shape (how many numbers per observation)
#
# Example (CartPole-v1):
#   [cart position, cart velocity, pole angle, pole angular vel]
# ============================================================

# Loop through each environment and inspect its observation space
for env_id in classic_control_envs:
    env = gym.make(env_id)                                  # Create environment

    print(f"Environment: {env_id}")                         # Print the env name
    print("Observation space:", env.observation_space)      # Print full space description

    # Most Classic Control envs use a Box (continuous) space.
    if isinstance(env.observation_space, gym.spaces.Box):
        print("Low values:", env.observation_space.low)     # Lower bound of each component
        print("High values:", env.observation_space.high)   # Upper bound of each component
        print("Shape:", env.observation_space.shape)        # Number of values in the obs vector

    print("-" * 60)                                         # Separator line for readability
    env.close()                                             # Release the environment

Environment: CartPole-v1
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Low values: [-4.8               -inf -0.41887903        -inf]
High values: [4.8               inf 0.41887903        inf]
Shape: (4,)
------------------------------------------------------------
Environment: MountainCar-v0
Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Low values: [-1.2  -0.07]
High values: [0.6  0.07]
Shape: (2,)
------------------------------------------------------------
Environment: MountainCarContinuous-v0
Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Low values: [-1.2  -0.07]
High values: [0.6  0.07]
Shape: (2,)
------------------------------------------------------------
Environment: Pendulum-v1
Observation space: Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)
Low values: [-1. -1. -8.]
High values: [1. 1. 8.]
Shape: (3,)
-----------------------------------------------

## 15. One Step Interaction Example

The next code cell demonstrates the **smallest possible RL interaction loop** using CartPole-v1.

**What the code does:**
1. Creates the environment.
2. Calls `env.reset(seed=42)` to obtain the initial state.
3. Samples one random action from `env.action_space`.
4. Calls `env.step(action)`, which returns the 5-tuple:
   `next_state, reward, terminated, truncated, info`.
5. Prints each component to show what an RL step looks like.
6. Closes the environment.

This is the basic loop that *every* RL algorithm wraps around — the difference is just **how the action is chosen** (random here, but learned in real RL).

In [24]:
# ============================================================
# CELL: One-Step Interaction Example (CartPole-v1)
# ------------------------------------------------------------
# This cell shows the smallest possible RL interaction loop.
# It demonstrates the core 5-tuple returned by env.step():
#
#   next_state, reward, terminated, truncated, info
#
# Steps:
#   1. Create the environment.
#   2. env.reset(seed=42) -> returns initial state (+ info).
#   3. Sample a random action.
#   4. env.step(action) -> returns the result of taking that
#      action: the new state, the reward, whether the
#      episode ended naturally (terminated), whether it was
#      cut off by a step/time limit (truncated), and any
#      extra info dict.
#   5. Print everything so we can see how an RL step looks.
#   6. Close the environment.
# ============================================================

env_id = "CartPole-v1"            # Environment to use for this example
env = gym.make(env_id)            # Create the environment

# Reset to start a fresh episode; seed=42 makes it reproducible
state, info = env.reset(seed=42)

# Show the initial state vector the agent receives
print("Initial state:")
print(state)

# Sample one random action from the action space (0 or 1)
action = env.action_space.sample()
print("\nAction taken:")
print(action)

# Apply the action -> get next state, reward, done flags, info
next_state, reward, terminated, truncated, info = env.step(action)

# Show the next state after the action
print("\nNext state:")
print(next_state)

# Show the reward returned for this step (+1 per step in CartPole)
print("\nReward:")
print(reward)

# Show the termination flags:
#   - terminated = True if the episode ended naturally (pole fell)
#   - truncated  = True if the episode was cut off due to time/limits
print("\nTerminated:", terminated)
print("Truncated:", truncated)

# Close the environment to free underlying resources
env.close()

Initial state:
[ 0.0273956  -0.00611216  0.03585979  0.0197368 ]

Action taken:
0

Next state:
[ 0.02727336 -0.20172954  0.03625453  0.32351476]

Reward:
1.0

Terminated: False
Truncated: False


## 16. Summary

In this notebook, we learned:

- How to load Gymnasium Classic Control environments
- How to inspect state space
- How to inspect action space
- Difference between discrete and continuous action spaces
- How to run a random agent
- How to record and display environment videos

This notebook prepares us for training RL agents later.

In [25]:
pip install gymnasium[box2d]

Note: you may need to restart the kernel to use updated packages.


In [26]:
import gymnasium as gym

# Create the lunar lander environment
env = gym.make("LunarLander-v3", render_mode="human")

# Reset the environment to start a new episode
observation, info = env.reset(seed=42)

# Run a loop where the agent takes random actions
for _ in range(1000):
    # Sample a random action from the discrete action space
    action = env.action_space.sample()  
    
    # Take a step in the environment
    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()

env.close()
